# PostgreSQL Internals — MVCC, WAL, VACUUM

Postgres looks simple from the outside — you send SQL, you get rows. Under the hood it runs a multi-version concurrency control system that lets thousands of transactions run simultaneously without locking each other out. Understanding MVCC, the Write-Ahead Log, and VACUUM is what separates a Staff DE from someone who just writes queries.

## MVCC — Multi-Version Concurrency Control

Every relation in Postgres stores not just the current row, but **multiple versions** of the same row. The system columns `xmin` and `xmax` are the MVCC fingerprint baked into every heap tuple:

- **`xmin`** — Transaction ID that *created* (inserted) this row version. The row is only visible to transactions with ID >= xmin.
- **`xmax`** — Transaction ID that *deleted or updated* this row. `xmax = 0` means the row is still alive — no one has deleted it yet.
- **Readers never block writers** — each transaction sees a *snapshot* of the database taken at its start time. Two concurrent writers each get their own version of a row.
- **Dead tuples accumulate** — on every `UPDATE`, Postgres inserts a new row version (`xmin = current txn`) and marks the old one as dead (`xmax = current txn`). Neither version is removed immediately.
- **No read locks on `SELECT`** — because readers use snapshot isolation, there is never a need to lock a row just to read it. This is the entire point of MVCC.

In [ ]:
from pathlib import Path
import sys
for _candidate in [Path('_setup'), Path('Basics/Databases/_setup')]:
    if _candidate.exists():
        sys.path.insert(0, str(_candidate.resolve()))
        break

from db_connections import get_postgres_conn
import pandas as pd

conn = get_postgres_conn()
cur = conn.cursor()

# Show xmin/xmax on real rows — the MVCC fingerprint
cur.execute("""
    SELECT
        xmin,
        xmax,
        endpoint_id,
        hostname,
        status
    FROM telemetry.endpoints
    LIMIT 5
""")
rows = cur.fetchall()
df = pd.DataFrame(rows, columns=['xmin','xmax','endpoint_id','hostname','status'])
print("MVCC row metadata (xmin=created by txn, xmax=deleted by txn):")
print(df.to_string(index=False))
print("\nxmax=0 means row is alive — no transaction has deleted it")

## WAL — Write-Ahead Log

WAL is Postgres's durability mechanism. Every single change to data goes through this log first:

- **WAL first, then data** — before Postgres modifies a data page on disk, it writes a WAL record describing the change. If the server crashes mid-write, Postgres replays the WAL on restart and the database is consistent.
- **Crash recovery** — at startup after a crash, Postgres walks backward through the WAL to find the last checkpoint, then replays all WAL records forward. No data is lost as long as WAL is intact.
- **Replication is WAL shipping** — physical replication works by streaming WAL bytes from primary to standby. The standby replays WAL continuously, staying in sync. Logical replication decodes WAL into row-level changes.
- **`pg_wal/` directory** — WAL files live here, each exactly 16 MB by default (`wal_segment_size`). Old segments are recycled or archived via `archive_command`.
- **`pg_lsn` (Log Sequence Number)** — a 64-bit offset into the WAL stream. `pg_current_wal_lsn()` gives the primary's current write position. The difference between primary and standby LSNs is the replication lag.

In [ ]:
# Show current WAL LSN and replication info
cur.execute("SELECT pg_current_wal_lsn() AS current_lsn")
lsn = cur.fetchone()[0]
print(f"Current WAL LSN: {lsn}")

# Show WAL size and configuration
cur.execute("""
    SELECT name, setting, unit
    FROM pg_settings
    WHERE name IN (
        'wal_level',
        'max_wal_size',
        'min_wal_size',
        'checkpoint_completion_target',
        'wal_buffers'
    )
    ORDER BY name
""")
df_wal = pd.DataFrame(cur.fetchall(), columns=['setting','value','unit'])
print("\nWAL configuration:")
print(df_wal.to_string(index=False))

## VACUUM — Dead Tuple Cleanup

MVCC's trade-off: old row versions pile up and must eventually be cleaned:

- **Dead tuples** — every `UPDATE` leaves a dead row version. Every `DELETE` marks a row dead. Without cleanup, these accumulate indefinitely, inflating table size and slowing scans.
- **`VACUUM`** — walks the table, marks dead tuples' space as reusable for future inserts. Does **not** shrink the physical file — the OS does not get the space back.
- **`VACUUM FULL`** — rewrites the entire table into a new heap file, compacting it. Returns disk space to the OS. Cost: an `ACCESS EXCLUSIVE` lock for the entire duration — all reads and writes on the table block. Use sparingly.
- **`autovacuum`** — background daemon that triggers VACUUM automatically based on dead tuple counts (`autovacuum_vacuum_threshold`, `autovacuum_vacuum_scale_factor`). You rarely need manual VACUUM.
- **Table bloat** — if autovacuum cannot keep up (high write rate, misconfigured thresholds), dead tuples accumulate faster than cleanup. The table grows larger than its live data. Symptoms: `n_dead_tup` is high, queries slow down.
- **`pg_stat_user_tables`** — the single most useful view for diagnosing bloat: shows live/dead tuple counts, last vacuum/analyze timestamps per table.

In [ ]:
cur.execute("""
    SELECT
        schemaname,
        tablename,
        n_live_tup,
        n_dead_tup,
        ROUND(n_dead_tup::numeric /
            NULLIF(n_live_tup + n_dead_tup, 0) * 100, 1) AS dead_pct,
        last_vacuum,
        last_autovacuum,
        last_analyze
    FROM pg_stat_user_tables
    WHERE schemaname = 'telemetry'
    ORDER BY n_dead_tup DESC
""")
df_vac = pd.DataFrame(cur.fetchall(),
    columns=['schema','table','live','dead','dead_pct',
             'last_vacuum','last_autovacuum','last_analyze'])
print("Table bloat report:")
print(df_vac.to_string(index=False))
print("\ndead_pct > 10% = consider manual VACUUM ANALYZE")
print("dead_pct > 20% = autovacuum is falling behind")

## Query Planner — How Postgres decides

The planner is a cost-based optimizer. It estimates the cheapest execution plan by:

- **Statistics** — `pg_statistic` stores histograms, most-common values, and null fractions per column. `ANALYZE` refreshes these. Stale statistics = bad row estimates = bad plans.
- **Access methods**: Postgres chooses between:
  - **Sequential Scan** — reads the entire heap. Preferred when selectivity is low (many rows match).
  - **Index Scan** — follows the index, fetches heap pages per match. Fast for high selectivity (few rows match), expensive for many matches due to random I/O.
  - **Bitmap Index Scan** — scans the index to build a bitmap of matching pages, then reads heap pages in physical order. A middle-ground — better than Index Scan when many rows match but still cheaper than Seq Scan.
- **Join strategies**: Hash Join (build hash on smaller side, probe with larger), Merge Join (both sides sorted), Nested Loop (small outer, indexed inner). Selected based on estimated row counts and memory (`work_mem`).
- **`ANALYZE`** — always run after a bulk load. Without fresh stats the planner uses defaults and picks terrible plans.
- **`EXPLAIN`** — shows the estimated plan. **`EXPLAIN ANALYZE`** actually executes the query and shows both estimated and actual row counts and timing — always use this when debugging a bad plan.

In [ ]:
# This is the most important query from R1 — now we read the plan properly
cur.execute("""
    EXPLAIN (ANALYZE, BUFFERS, FORMAT TEXT)
    SELECT
        e.service_type,
        ROUND(AVG(m.value)::numeric, 2) AS avg_cpu,
        COUNT(*) AS sample_count
    FROM telemetry.metrics m
    JOIN telemetry.endpoints e ON e.endpoint_id = m.endpoint_id
    WHERE
        m.metric_name = 'cpu_percent'
        AND m.recorded_at >= NOW() - INTERVAL '7 days'
    GROUP BY e.service_type
    ORDER BY avg_cpu DESC
""")
plan = cur.fetchall()
print("EXPLAIN ANALYZE output:")
for row in plan:
    print(row[0])

## Reading the EXPLAIN output — field by field

| Term | What it means |
|------|---------------|
| `cost=X..Y` | Estimated startup cost .. total cost (in arbitrary planner units) |
| `rows=N` | Planner's estimated output row count |
| `actual time=X..Y` | Real milliseconds: time to first row .. time to last row |
| `actual rows=N` | Real row count — compare to estimated to spot mis-estimates |
| `loops=N` | How many times this plan node executed (e.g. nested loop iterations) |
| `Seq Scan` | Full table scan — no index used; reads every heap page |
| `Index Scan` | Index used; heap page fetched for each matching index entry |
| `Bitmap Heap Scan` | Index scanned to build a page bitmap, then heap pages read in order |
| `Hash Join` | Build hash table on smaller relation, probe it with each row from larger |
| `Buffers: shared hit=N` | Pages served from shared buffer cache (free — no disk I/O) |
| `Buffers: shared read=N` | Pages read from disk (expensive — look for this when tuning) |

**The key diagnostic**: compare `rows=N` (estimated) vs `actual rows=N` (real). A big gap means stale statistics. Run `ANALYZE tablename` and re-check.

In [ ]:
# The seq scan on metrics is the bottleneck — add the index we identified in R1
cur.execute("""
    CREATE INDEX IF NOT EXISTS idx_metrics_name_time
    ON telemetry.metrics (metric_name, recorded_at DESC)
""")
conn.commit()
print("Index created: idx_metrics_name_time on (metric_name, recorded_at DESC)")

# Re-run EXPLAIN ANALYZE — should now show Index Scan
cur.execute("""
    EXPLAIN (ANALYZE, BUFFERS, FORMAT TEXT)
    SELECT
        e.service_type,
        ROUND(AVG(m.value)::numeric, 2) AS avg_cpu,
        COUNT(*) AS sample_count
    FROM telemetry.metrics m
    JOIN telemetry.endpoints e ON e.endpoint_id = m.endpoint_id
    WHERE
        m.metric_name = 'cpu_percent'
        AND m.recorded_at >= NOW() - INTERVAL '7 days'
    GROUP BY e.service_type
    ORDER BY avg_cpu DESC
""")
plan2 = cur.fetchall()
print("\nEXPLAIN ANALYZE after index:")
for row in plan2:
    print(row[0])
print("\nCompare execution time before vs after.")
print("Look for 'Index Scan using idx_metrics_name_time'")

## Partitioning — splitting large tables

Partitioning lets Postgres treat one logical table as multiple physical heaps:

- **Range partitioning by `recorded_at`** — January data lives in `metrics_2026_01`, February in `metrics_2026_02`. A query with `WHERE recorded_at >= '2026-02-01'` only touches `metrics_2026_02`.
- **Partition pruning** — the planner reads the `WHERE` clause and skips partitions that cannot possibly satisfy the filter. A 12-month table becomes a 1-month scan automatically. Requires `enable_partition_pruning = on` (default).
- **`pg_partman`** — extension that manages partition creation automatically. Configure it once: `CALL partman.create_parent('telemetry.metrics', 'recorded_at', 'range', 'monthly')` and new monthly partitions appear automatically.
- **Constraint exclusion** — for older Postgres or non-`pg_partman` setups, `CHECK` constraints on each partition tell the planner which partitions to skip. Less automatic but same pruning effect.
- **When to partition**: table > 100 GB AND queries **always** filter by the partition key. If you query across all partitions often, partitioning adds overhead. The partition key must appear in `WHERE` for pruning to kick in.
- **Index per partition** — each partition has its own indexes. Global indexes across all partitions are not supported in standard range/list/hash partitioning.

In [ ]:
# Check if metrics is partitioned (it's not — this shows the contrast)
cur.execute("""
    SELECT
        schemaname,
        tablename,
        pg_size_pretty(pg_total_relation_size(schemaname||'.'||tablename)) AS total_size,
        pg_size_pretty(pg_relation_size(schemaname||'.'||tablename)) AS table_size,
        pg_size_pretty(pg_indexes_size(schemaname||'.'||tablename)) AS index_size
    FROM pg_tables
    WHERE schemaname = 'telemetry'
    ORDER BY pg_total_relation_size(schemaname||'.'||tablename) DESC
""")
df_size = pd.DataFrame(cur.fetchall(),
    columns=['schema','table','total_size','table_size','index_size'])
print("Table sizes in telemetry schema:")
print(df_size.to_string(index=False))
print("\nAt 500K rows metrics is small — partitioning pays off at 50M+ rows")
print("Design: PARTITION BY RANGE (recorded_at) — one partition per month")

## Key takeaways — what interviewers actually ask

**"How does Postgres handle concurrent reads and writes?"**  
→ MVCC — readers never block writers. Each transaction sees a snapshot of the database taken at its start time. There are no read locks on `SELECT` ever. Writers create new row versions; readers see the old ones until they commit.

**"What is WAL and why does it matter?"**  
→ Durability + replication. Every write goes to WAL first, before touching the actual data pages. Crash recovery replays WAL from the last checkpoint. Standbys stream WAL from primary and replay it continuously. WAL level `logical` enables logical replication and CDC (Change Data Capture).

**"What is table bloat and how do you fix it?"**  
→ Dead tuples from MVCC accumulate on every `UPDATE`/`DELETE`. `VACUUM` marks dead space as reusable but does not shrink the file. `VACUUM FULL` rewrites the table and reclaims disk space, but holds an exclusive lock. `autovacuum` handles routine cleanup automatically — manual `VACUUM` when autovacuum falls behind (check `pg_stat_user_tables.n_dead_tup`).

**"Why is my query doing a seq scan when I have an index?"**  
→ The planner estimates that an index scan costs more than a seq scan at low selectivity (many rows match — random I/O on every match is expensive). Check `EXPLAIN ANALYZE`: compare estimated vs actual rows. If estimates are wrong, run `ANALYZE` to refresh statistics. If selectivity is genuinely low, a seq scan may actually be correct.

**"When would you partition a table?"**  
→ When the table is very large (100 GB+) AND queries consistently filter by the partition key. Partition pruning skips irrelevant partitions entirely, turning a full-table scan into a single-partition scan. If queries scan across all time ranges, partitioning adds overhead without benefit.

---
*Simplicity and clarity is Gold.*